In [ ]:
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.passes import CheckMap
from qiskit.circuit.controlflow import ForLoopOp, WhileLoopOp, IfElseOp

def circuit_features(qc):
    ops = {inst.operation.name for inst in qc.data}
    has_mid_meas = any(op.name == "measure" and i < len(qc.data)-1 for i, op in enumerate([d.operation for d in qc.data]))
    has_reset = "reset" in ops
    has_conditional = any(getattr(inst.operation, "condition", None) for inst in qc.data)
    has_control_flow = any(isinstance(inst.operation, (ForLoopOp, WhileLoopOp, IfElseOp)) for inst in qc.data)
    twoq_pairs = {(qargs[0].index, qargs[1].index) for inst, qargs, _ in qc.data if inst.name in {"cx","cz","swap"} and len(qargs) == 2}
    return {
        "ops": ops,
        "has_mid_meas": has_mid_meas,
        "has_reset": has_reset,
        "has_conditional": has_conditional,
        "has_control_flow": has_control_flow,
        "twoq_pairs": twoq_pairs,
        "n_qubits": qc.num_qubits,
    }

def backend_supports_dynamic(backend):
    # IBM example: dynamic circuits flag lives in backend.configuration() or backend.target
    try:
        cfg = backend.configuration()
        return getattr(cfg, "supports_mid_circuit_measurement", False) or getattr(cfg, "dynamic_circuits", False)
    except Exception:
        # Aer simulators generally support dynamic in software
        name = backend.name().lower()
        return "aer" in name or "simulator" in name

def is_candidate_backend(qc, backend):
    feats = circuit_features(qc)
    # Qubit count
    try:
        if feats["n_qubits"] > backend.configuration().num_qubits:
            return False
    except Exception:
        pass  # Simulators may not expose this nicely

    # Dynamic requirements
    if any([feats["has_mid_meas"], feats["has_reset"], feats["has_conditional"], feats["has_control_flow"]]):
        if not backend_supports_dynamic(backend):
            return False

    # Basis gates
    try:
        target_ops = set(getattr(backend, "target").operation_names)
    except Exception:
        # Fallback: use configuration().basis_gates if present; else trust transpiler
        target_ops = set(getattr(getattr(backend, "configuration", lambda: type("X", (), {}) )(), "basis_gates", []))

    if target_ops:
        # If the circuit has unknown ops, the transpiler may still decompose them.
        # We only hard-fail if there are exotic ops and no standard basis present.
        if not {"measure","reset","barrier"}.issuperset(feats["ops"]):
            pass  # let transpile decide

    # Connectivity quick check (optional prefilter)
    try:
        cmap = CouplingMap(backend.configuration().coupling_map)
        # If there is a 2q gate between qubits that are completely disconnected even after routing,
        # transpiler will fail; we let transpile be the final arbiter.
    except Exception:
        pass

    # Final proof: attempt a transpile
    try:
        transpile(qc, backend, optimization_level=1)
        return True
    except Exception:
        return False

def recommend_backends(qc, providers, limit=5):
    candidates = []
    for prov in providers:
        for backend in prov.backends():
            try:
                if is_candidate_backend(qc, backend):
                    candidates.append(backend)
            except Exception:
                continue
    # Simple ranking: prefer simulators first for safety, then hardware by qubit count asc
    simulators = [b for b in candidates if "simulator" in b.name().lower()]
    hardware = [b for b in candidates if "simulator" not in b.name().lower()]
    hardware.sort(key=lambda b: getattr(getattr(b, "configuration", lambda: type("X", (), {}) )(), "num_qubits", 999))
    ordered = simulators + hardware
    return ordered[:limit]
